<a href="https://colab.research.google.com/github/shukrullo24/Python-darslari/blob/main/komakchi_admin_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install aiogram

import asyncio
import logging
from aiogram import Bot, Dispatcher, types, F
from aiogram.filters import Command
from aiogram.fsm.context import FSMContext
from aiogram.fsm.state import State, StatesGroup
from aiogram.types import ReplyKeyboardMarkup, KeyboardButton, ReplyKeyboardRemove

# --- SOZLAMALAR ---
API_TOKEN = '8793631631:AAEfPPMRGdIdukPrw13J-ysQwfy8ZL_6KVU'
ADMIN_ID = 7278564311 # @userinfobot orqali olingan ID

logging.basicConfig(level=logging.INFO)
bot = Bot(token=API_TOKEN)
dp = Dispatcher()

# Savollar holati (Yangi qadamlar qo'shildi)
class Registration(StatesGroup):
    name = State()
    phone = State()
    location = State()
    has_certificate = State() # Sertifikat bormi?
    cert_level = State()      # Darajasi qanaqa?

# 1. Start
@dp.message(Command("start"))
async def cmd_start(message: types.Message, state: FSMContext):
    await message.answer("Xush kelibsiz! Koreyada o'qish bo'yicha murojaat qoldirish uchun ismingizni kiriting:")
    await state.set_state(Registration.name)

# 2. Ism
@dp.message(Registration.name)
async def process_name(message: types.Message, state: FSMContext):
    await state.update_data(name=message.text)
    kb = [[KeyboardButton(text="📞 Telefon raqamni yuborish", request_contact=True)]]
    keyboard = ReplyKeyboardMarkup(keyboard=kb, resize_keyboard=True, one_time_keyboard=True)
    await message.answer(f"Rahmat. Endi telefon raqamingizni yuboring:", reply_markup=keyboard)
    await state.set_state(Registration.phone)

# 3. Telefon
@dp.message(Registration.phone, F.contact | F.text)
async def process_phone(message: types.Message, state: FSMContext):
    phone = message.contact.phone_number if message.contact else message.text
    await state.update_data(phone=phone)
    await message.answer("Hozirda qaysi shahar/viloyatda yashaysiz?", reply_markup=ReplyKeyboardRemove())
    await state.set_state(Registration.location)

# 4. Manzil
@dp.message(Registration.location)
async def process_location(message: types.Message, state: FSMContext):
    await state.update_data(location=message.text)

    # Sertifikat bor-yo'qligini so'rash
    kb = [
        [KeyboardButton(text="Bor ✅"), KeyboardButton(text="Yo'q ❌")]
    ]
    keyboard = ReplyKeyboardMarkup(keyboard=kb, resize_keyboard=True, one_time_keyboard=True)
    await message.answer("Sizda til sertifikati (TOPIK, IELTS, va h.k.) bormi?", reply_markup=keyboard)
    await state.set_state(Registration.has_certificate)

# 5. Sertifikat bormi yoki yo'q?
@dp.message(Registration.has_certificate)
async def process_cert_check(message: types.Message, state: FSMContext):
    if message.text == "Bor ✅":
        await message.answer("Qaysi sertifikat va nechanchi daraja? (Masalan: TOPIK 4 yoki IELTS 6.5)", reply_markup=ReplyKeyboardRemove())
        await state.set_state(Registration.cert_level)
    else:
        await state.update_data(cert_level="Mavjud emas")
        await finish_registration(message, state)

# 6. Sertifikat darajasini qabul qilish
@dp.message(Registration.cert_level)
async def process_cert_level(message: types.Message, state: FSMContext):
    await state.update_data(cert_level=message.text)
    await finish_registration(message, state)

# Yakunlash funksiyasi
async def finish_registration(message: types.Message, state: FSMContext):
    user_data = await state.get_data()

    admin_text = (
        "🚀 **Yangi murojaat!**\n\n"
        f"👤 **Ismi:** {user_data['name']}\n"
        f"📞 **Tel:** {user_data['phone']}\n"
        f"📍 **Manzil:** {user_data['location']}\n"
        f"📜 **Sertifikat:** {user_data['cert_level']}\n"
        f"🔗 **User:** @{message.from_user.username or 'link yoq'}"
    )

    await bot.send_message(ADMIN_ID, admin_text, parse_mode="Markdown")
    await message.answer("Rahmat! Ma'lumotlaringiz qabul qilindi. Tez orada mutaxassislarimiz bog'lanishadi.", reply_markup=ReplyKeyboardRemove())
    await state.clear()

async def main():
    await dp.start_polling(bot)

if __name__ == "__main__":
    asyncio.run(main())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 727.3/727.3 kB 11.5 MB/s eta 0:00:00


RuntimeError: asyncio.run() cannot be called from a running event loop